<a href="https://colab.research.google.com/github/jaywong8/Detection-of-Atrial-Fibrillation-Termination/blob/main/Agamjot_cnn.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount("/content/drive")
!pip install -q wfdb
%cd /content
!rm -rf Detection-of-Atrial-Fibrillation-Termination && git clone -q https://github.com/jaywong8/Detection-of-Atrial-Fibrillation-Termination.git
%cd /content/Detection-of-Atrial-Fibrillation-Termination

import wfdb, numpy as np, pandas as pd, time
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed

FS, N = 250, 2500
DATA = Path("data/10s-30s")
CACHE = Path("/content/drive/MyDrive/af_cnn_cache"); CACHE.mkdir(parents=True, exist_ok=True)

def pn_dir(stem):
    pid = stem.split("_")[0]
    return f"icentia11k-continuous-ecg/1.0/{pid[:3]}/{pid}"

def grab(stem, grp):
    out = []
    for _, r in grp.iterrows():
        s0 = int(r.input_start_sec * FS)
        try:
            sig = wfdb.rdrecord(stem, pn_dir=pn_dir(stem), sampfrom=s0, sampto=s0 + N, channels=[0]).p_signal[:, 0]
            if len(sig) == N and np.isfinite(sig).all(): out.append((r.window_id, sig.astype(np.float32)))
        except Exception:
            pass
    return out

def cache_split(name, file):
    if (CACHE / f"{name}.npz").exists(): print(name, "already cached"); return
    df = pd.read_csv(DATA / file); t = time.time(); got = []
    groups = list(df.groupby("record_stem"))
    with ThreadPoolExecutor(12) as ex:
        futs = [ex.submit(grab, s, g) for s, g in groups]
        for i, f in enumerate(as_completed(futs), 1):
            got += f.result()
            if i % 200 == 0 or i == len(futs): print(f"{name}: {i}/{len(futs)} records, {len(got)} windows, {time.time()-t:.0f}s")
    ids = np.array([g[0] for g in got]); X = np.stack([g[1] for g in got])
    y = df.set_index("window_id").loc[ids, "label"].values.astype(np.int64)
    np.savez(CACHE / f"{name}.npz", X=X, y=y, ids=ids)
    print(f"{name} saved: {X.shape}, {len(df)-len(ids)} dropped")

for name, file in [("validation", "validation_windows.csv"), ("test", "test_windows.csv"), ("train", "train_windows_patient_episode_capped_natural.csv")]:
    cache_split(name, file)

Mounted at /content/drive
/content
/content/Detection-of-Atrial-Fibrillation-Termination
validation already cached
test already cached
train already cached


In [2]:
import torch, torch.nn as nn, json, copy
from sklearn.metrics import roc_auc_score, average_precision_score
dev = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", dev)

class Block(nn.Module):
    def __init__(s, cin, cout, stride, k=7, drop=0.2):
        super().__init__()
        s.c1 = nn.Conv1d(cin, cout, k, stride, k//2, bias=False); s.b1 = nn.BatchNorm1d(cout)
        s.c2 = nn.Conv1d(cout, cout, k, 1, k//2, bias=False); s.b2 = nn.BatchNorm1d(cout)
        s.dr = nn.Dropout(drop); s.act = nn.ReLU()
        s.skip = nn.Sequential() if (cin == cout and stride == 1) else nn.Sequential(nn.Conv1d(cin, cout, 1, stride, bias=False), nn.BatchNorm1d(cout))
    def forward(s, x):
        h = s.dr(s.act(s.b1(s.c1(x)))); h = s.b2(s.c2(h))
        return s.act(h + s.skip(x))

class ResNet1D(nn.Module):
    def __init__(s, width=32, drop=0.2):
        super().__init__()
        w = width
        s.stem = nn.Sequential(nn.Conv1d(1, w, 15, 2, 7, bias=False), nn.BatchNorm1d(w), nn.ReLU())
        s.blocks = nn.Sequential(Block(w, w, 2, drop=drop), Block(w, 2*w, 2, drop=drop), Block(2*w, 2*w, 2, drop=drop), Block(2*w, 4*w, 2, drop=drop))
        s.head = nn.Sequential(nn.AdaptiveAvgPool1d(1), nn.Flatten(), nn.Dropout(drop), nn.Linear(4*w, 1))
    def forward(s, x): return s.head(s.blocks(s.stem(x))).squeeze(1)

def normalize(X):
    X = X - np.median(X, axis=1, keepdims=True)
    return (X / (X.std(axis=1, keepdims=True) + 1e-6)).astype(np.float32)

def augment(xb):
    xb = xb * torch.empty(xb.size(0), 1, 1, device=xb.device).uniform_(0.8, 1.2)
    xb = xb + 0.05 * torch.randn_like(xb)
    return torch.roll(xb, int(torch.randint(-125, 126, (1,))), dims=2)

def predict(model, X, bs=512):
    model.eval(); out = []
    with torch.no_grad():
        for i in range(0, len(X), bs):
            out.append(torch.sigmoid(model(torch.from_numpy(X[i:i+bs]).unsqueeze(1).to(dev))).cpu().numpy())
    return np.concatenate(out)

def train_one(cfg, seed, Xtr, ytr, Xva, yva):
    torch.manual_seed(seed); np.random.seed(seed)
    model = ResNet1D(cfg["width"], cfg["dropout"]).to(dev)
    opt = torch.optim.AdamW(model.parameters(), lr=cfg["lr"], weight_decay=cfg["weight_decay"])
    sched = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, mode="max", factor=0.5, patience=3)
    pos_w = torch.tensor((ytr == 0).sum() / max((ytr == 1).sum(), 1), device=dev) if cfg["class_weight"] else None
    lossf = nn.BCEWithLogitsLoss(pos_weight=pos_w)
    Xt, yt = torch.from_numpy(Xtr), torch.from_numpy(ytr.astype(np.float32))
    best, best_state, bad = -1, None, 0
    for ep in range(cfg["max_epochs"]):
        model.train(); perm = torch.randperm(len(Xt))
        for i in range(0, len(Xt), cfg["batch_size"]):
            idx = perm[i:i+cfg["batch_size"]]
            xb = Xt[idx].unsqueeze(1).to(dev); yb = yt[idx].to(dev)
            if cfg["augment"]: xb = augment(xb)
            opt.zero_grad(); loss = lossf(model(xb), yb); loss.backward(); opt.step()
        p = predict(model, Xva); auc = roc_auc_score(yva, p); sched.step(auc)
        if auc > best: best, best_state, bad = auc, copy.deepcopy(model.state_dict()), 0
        else: bad += 1
        print(f"  seed {seed} ep {ep+1:2d}  val ROC-AUC {auc:.3f}  PR-AUC {average_precision_score(yva, p):.3f}")
        if bad >= cfg["patience"]: break
    model.load_state_dict(best_state)
    return model, best

device: cuda


In [3]:
tr, va, te = (np.load(CACHE / f"{n}.npz", allow_pickle=True) for n in ["train", "validation", "test"])
Xtr, ytr, Xva, yva, Xte = normalize(tr["X"]), tr["y"], normalize(va["X"]), va["y"], normalize(te["X"])
print("train", Xtr.shape, "val", Xva.shape, "test", Xte.shape)

BASE = dict(width=32, dropout=0.2, lr=1e-3, weight_decay=1e-3, batch_size=64, max_epochs=40, patience=8, augment=True, class_weight=True)
CONFIGS = {"cnn_base": BASE, "cnn_lowlr": {**BASE, "lr": 3e-4}, "cnn_noaug": {**BASE, "augment": False}}
SEEDS = [0, 1, 2]

OUT = CACHE / "results"; (OUT / "preds").mkdir(parents=True, exist_ok=True); (OUT / "configs").mkdir(exist_ok=True); (OUT / "weights").mkdir(exist_ok=True)
summary = []
for cname, cfg in CONFIGS.items():
    json.dump({**cfg, "seeds": SEEDS, "input": "10s raw ECG, 250Hz, per-window median/std normalized"}, open(OUT / "configs" / f"{cname}.json", "w"), indent=2)
    for seed in SEEDS:
        print(f"\n=== {cname} seed {seed}")
        model, best = train_one(cfg, seed, Xtr, ytr, Xva, yva)
        tag = f"{cname}_s{seed}"
        pd.DataFrame({"window_id": va["ids"], "probability": predict(model, Xva)}).to_csv(OUT / "preds" / f"{tag}_validation.csv", index=False)
        pd.DataFrame({"window_id": te["ids"], "probability": predict(model, Xte)}).to_csv(OUT / "preds" / f"{tag}_test.csv", index=False)
        torch.save(model.state_dict(), OUT / "weights" / f"{tag}.pt")
        summary.append({"config": cname, "seed": seed, "val_ROC_AUC": best})

s = pd.DataFrame(summary)
print("\n", s.groupby("config")["val_ROC_AUC"].agg(["mean", "std"]).round(3))

train (8724, 2500) val (3000, 2500) test (3000, 2500)

=== cnn_base seed 0
  seed 0 ep  1  val ROC-AUC 0.624  PR-AUC 0.278
  seed 0 ep  2  val ROC-AUC 0.632  PR-AUC 0.297
  seed 0 ep  3  val ROC-AUC 0.633  PR-AUC 0.279
  seed 0 ep  4  val ROC-AUC 0.623  PR-AUC 0.273
  seed 0 ep  5  val ROC-AUC 0.649  PR-AUC 0.313
  seed 0 ep  6  val ROC-AUC 0.674  PR-AUC 0.328
  seed 0 ep  7  val ROC-AUC 0.655  PR-AUC 0.308
  seed 0 ep  8  val ROC-AUC 0.652  PR-AUC 0.328
  seed 0 ep  9  val ROC-AUC 0.617  PR-AUC 0.269
  seed 0 ep 10  val ROC-AUC 0.659  PR-AUC 0.302
  seed 0 ep 11  val ROC-AUC 0.649  PR-AUC 0.309
  seed 0 ep 12  val ROC-AUC 0.685  PR-AUC 0.325
  seed 0 ep 13  val ROC-AUC 0.668  PR-AUC 0.300
  seed 0 ep 14  val ROC-AUC 0.687  PR-AUC 0.324
  seed 0 ep 15  val ROC-AUC 0.637  PR-AUC 0.290
  seed 0 ep 16  val ROC-AUC 0.657  PR-AUC 0.288
  seed 0 ep 17  val ROC-AUC 0.675  PR-AUC 0.324
  seed 0 ep 18  val ROC-AUC 0.668  PR-AUC 0.314
  seed 0 ep 19  val ROC-AUC 0.669  PR-AUC 0.296
  seed 0 ep 2

In [4]:
P = OUT / "preds"
for cname in CONFIGS:
    for split in ["validation", "test"]:
        dfs = [pd.read_csv(P / f"{cname}_s{s}_{split}.csv") for s in SEEDS]
        ens = dfs[0][["window_id"]].copy()
        ens["probability"] = np.mean([d["probability"].values for d in dfs], axis=0)
        ens.to_csv(P / f"{cname}_ensemble_{split}.csv", index=False)
    e = pd.read_csv(P / f"{cname}_ensemble_validation.csv")
    print(f"{cname} ensemble  val ROC-AUC {roc_auc_score(yva, e['probability']):.3f}  PR-AUC {average_precision_score(yva, e['probability']):.3f}")
print("\nfiles:", sorted(p.name for p in P.iterdir()))

cnn_base ensemble  val ROC-AUC 0.691  PR-AUC 0.336
cnn_lowlr ensemble  val ROC-AUC 0.700  PR-AUC 0.338
cnn_noaug ensemble  val ROC-AUC 0.673  PR-AUC 0.321

files: ['cnn_base_ensemble_test.csv', 'cnn_base_ensemble_validation.csv', 'cnn_base_s0_test.csv', 'cnn_base_s0_validation.csv', 'cnn_base_s1_test.csv', 'cnn_base_s1_validation.csv', 'cnn_base_s2_test.csv', 'cnn_base_s2_validation.csv', 'cnn_lowlr_ensemble_test.csv', 'cnn_lowlr_ensemble_validation.csv', 'cnn_lowlr_s0_test.csv', 'cnn_lowlr_s0_validation.csv', 'cnn_lowlr_s1_test.csv', 'cnn_lowlr_s1_validation.csv', 'cnn_lowlr_s2_test.csv', 'cnn_lowlr_s2_validation.csv', 'cnn_noaug_ensemble_test.csv', 'cnn_noaug_ensemble_validation.csv', 'cnn_noaug_s0_test.csv', 'cnn_noaug_s0_validation.csv', 'cnn_noaug_s1_test.csv', 'cnn_noaug_s1_validation.csv', 'cnn_noaug_s2_test.csv', 'cnn_noaug_s2_validation.csv']


In [5]:
import shutil
shutil.copytree(OUT / "preds", "/content/upload/results/preds", dirs_exist_ok=True)
shutil.copytree(OUT / "configs", "/content/upload/results/configs", dirs_exist_ok=True)
shutil.make_archive("/content/cnn_results", "zip", "/content/upload")
from google.colab import files
files.download("/content/cnn_results.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>